# Research Question 7

How does the volume of food-inflation-related media coverage change during periods of lower and higher inflation?

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
FRED_FILE = "fred_cpi_germany.json"   # local JSON file with CPI time series
GDELT_FILE = "gdelt_API.json"         # local JSON file with GDELT article data

START_DATE = "2020-01-01"             # start of the analysis period
END_DATE = "2024-12-31"               # end of the analysis period

HIGH_INFLATION_QUANTILE = 0.75        # top 25% of inflation values are treated as high inflation
MIN_ARTICLES_PER_MONTH = 1            # keep only months with at least 1 article

In [3]:
def load_fred_json(filepath):
    """
    Load CPI data from a local JSON file and return it as a DataFrame.

    The FRED JSON file is expected to store a mapping from date to CPI value.
    This mapping is converted into a two-column pandas DataFrame.
    """

    # Read the JSON file into a Python dictionary.
    with open(filepath, "r", encoding="utf-8") as file:
        raw_data = json.load(file)

    # Convert dictionary items into rows with explicit column names.
    dataframe = pd.DataFrame(
        list(raw_data.items()),
        columns=["date", "cpi"],
    )

    return dataframe


def load_gdelt_json(filepath):
    """
    Load GDELT article data from a local JSON file and return it as a DataFrame.

    The GDELT JSON file is expected to contain a list of dictionaries,
    where each dictionary represents one article.
    """

    # Read the JSON file into a Python list of article dictionaries.
    with open(filepath, "r", encoding="utf-8") as file:
        raw_data = json.load(file)

    # Convert the list directly into a tabular pandas structure.
    dataframe = pd.DataFrame(raw_data)

    return dataframe             # return the DataFrame

## Data Cleaning Functions

The following functions clean the CPI data and the GDELT article data.

The cleaning process includes:
- converting dates to datetime format
- converting values to numeric format where needed
- removing missing values
- removing duplicates
- sorting the data
- filtering the selected time period

In [4]:
def clean_cpi_data(dataframe):
    """
    Clean CPI data and restrict it to the selected analysis period.
    """

    # Work on a copy to avoid modifying the original DataFrame.
    cleaned = dataframe.copy()

    # Convert date and CPI columns to usable analysis formats.
    cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
    cleaned["cpi"] = pd.to_numeric(cleaned["cpi"], errors="coerce")

    # Remove invalid rows, duplicate dates, and sort the time series.
    cleaned = cleaned.dropna(subset=["date", "cpi"])
    cleaned = cleaned.drop_duplicates(subset=["date"])
    cleaned = cleaned.sort_values("date").reset_index(drop=True)

    # Keep only observations inside the predefined analysis window.
    cleaned = cleaned[
        (cleaned["date"] >= START_DATE) &
        (cleaned["date"] <= END_DATE)
    ].copy()

    return cleaned


def clean_gdelt_data(dataframe):
    """
    Clean GDELT article data and keep only relevant observations.
    """

    # Work on a copy to avoid changing the original input.
    cleaned = dataframe.copy()

    # Keep only columns that are relevant for article-level analysis.
    useful_columns = ["title", "seendate", "language", "sourcecountry", "domain"]
    available_columns = [column for column in useful_columns if column in cleaned.columns]
    cleaned = cleaned[available_columns].copy()

    # Standardize titles and convert GDELT timestamps to datetime format.
    cleaned["title"] = cleaned["title"].astype(str).str.strip()
    cleaned["seendate"] = pd.to_datetime(
        cleaned["seendate"],
        errors="coerce",
        format="%Y%m%dT%H%M%SZ"
    )

    # Remove invalid, empty, and duplicate article entries.
    cleaned = cleaned.dropna(subset=["title", "seendate"])
    cleaned = cleaned[cleaned["title"] != ""]
    cleaned = cleaned.drop_duplicates(subset=["title", "seendate"])
    cleaned = cleaned.sort_values("seendate").reset_index(drop=True)

    # Restrict the data to the selected analysis period.
    cleaned = cleaned[
        (cleaned["seendate"] >= START_DATE) &
        (cleaned["seendate"] <= END_DATE)
    ].copy()

    return cleaned

In [5]:
def add_cpi_features(dataframe):
    """
    Add monthly time features and inflation measures to CPI data.
    """

    # Work on a copy to avoid modifying the original DataFrame.
    result = dataframe.copy()

    # Create time-based features for later monthly merging and analysis.
    result["year"] = result["date"].dt.year
    result["month_number"] = result["date"].dt.month
    result["month"] = result["date"].dt.to_period("M").dt.to_timestamp()

    # Calculate monthly and yearly inflation rates from the CPI series.
    result["inflation_mom"] = result["cpi"].pct_change(periods=1) * 100
    result["inflation_yoy"] = result["cpi"].pct_change(periods=12) * 100

    return result


def add_gdelt_time_features(dataframe):
    """
    Add monthly time features to article-level GDELT data.
    """

    # Work on a copy to avoid modifying the original DataFrame.
    result = dataframe.copy()

    # Extract year and month information from the article timestamp.
    result["year"] = result["seendate"].dt.year
    result["month_number"] = result["seendate"].dt.month
    result["month"] = result["seendate"].dt.to_period("M").dt.to_timestamp()

    return result


def aggregate_articles_monthly(dataframe):
    """
    Aggregate article-level data to monthly article counts.
    """

    # Count how many articles are available in each month.
    monthly = (
        dataframe.groupby("month", as_index=False)
        .agg(article_count=("title", "count"))
        .sort_values("month")
        .reset_index(drop=True)
    )

    return monthly


def assign_inflation_regime(dataframe, quantile=0.75):
    """
    Classify months into lower- and high-inflation periods
    based on a quantile threshold of yearly inflation.
    """

    # Work on a copy and compute the threshold for high inflation.
    result = dataframe.copy()
    threshold = result["inflation_yoy"].quantile(quantile)

    # Label months above the threshold as high inflation.
    result["inflation_regime"] = "lower_inflation"
    result.loc[
        result["inflation_yoy"] >= threshold,
        "inflation_regime"
    ] = "high_inflation"

    return result, threshold                       # return DataFrame and threshold value

In [6]:
# Load the raw CPI and GDELT datasets from the local JSON files.
cpi_raw = load_fred_json(FRED_FILE)
gdelt_raw = load_gdelt_json(GDELT_FILE)

# Print the original dataset dimensions as a quick data-loading check.
print("CPI raw shape:", cpi_raw.shape)
print("GDELT raw shape:", gdelt_raw.shape)

CPI raw shape: (843, 2)
GDELT raw shape: (100, 8)


In [7]:
# Clean both raw datasets before further analysis.
cpi_clean = clean_cpi_data(cpi_raw)
gdelt_clean = clean_gdelt_data(gdelt_raw)

# Print the cleaned dataset dimensions to verify the effect of preprocessing.
print("CPI cleaned shape:", cpi_clean.shape)
print("GDELT cleaned shape:", gdelt_clean.shape)

CPI cleaned shape: (60, 2)
GDELT cleaned shape: (100, 5)


In [8]:
# Add time- and inflation-related features to the cleaned datasets.
cpi_features = add_cpi_features(cpi_clean)
gdelt_features = add_gdelt_time_features(gdelt_clean)

# Aggregate article-level GDELT data to monthly article counts.
gdelt_monthly = aggregate_articles_monthly(gdelt_features)

# Check the size of the monthly article table and inspect the first rows.
print("Monthly article data shape:", gdelt_monthly.shape)

gdelt_monthly.head()

Monthly article data shape: (4, 2)


,month,article_count
0,2023-10-01,30
1,2023-11-01,33
2,2023-12-01,34
3,2024-01-01,3


In [9]:
# Merge monthly CPI features with monthly article counts.
analysis_df = pd.merge(
    cpi_features[["month", "date", "cpi", "inflation_mom", "inflation_yoy"]],
    gdelt_monthly,
    on="month",
    how="inner"
)

# Remove incomplete rows, keep only months with article coverage,
# and sort the final analysis table chronologically.
analysis_df = analysis_df.dropna(subset=["inflation_yoy"])
analysis_df = analysis_df[
    analysis_df["article_count"] >= MIN_ARTICLES_PER_MONTH
].copy()
analysis_df = analysis_df.sort_values("month").reset_index(drop=True)

# Classify months into lower- and high-inflation periods
# using the predefined quantile threshold.
analysis_df, inflation_threshold = assign_inflation_regime(
    analysis_df,
    quantile=HIGH_INFLATION_QUANTILE
)

# Print the inflation threshold and inspect the final analysis dataset.
print("Inflation threshold (YoY):", round(inflation_threshold, 2))
print("Final analysis shape:", analysis_df.shape)

analysis_df.head()

Inflation threshold (YoY): 3.73
Final analysis shape: (4, 7)


,month,date,cpi,inflation_mom,inflation_yoy,article_count,inflation_regime
0,2023-10-01,2023-10-01,124.1946,0.000000,3.788530,30,high_inflation
1,2023-11-01,2023-11-01,123.6675,-0.424415,3.166208,33,lower_inflation
2,2023-12-01,2023-12-01,123.7729,0.085229,3.710255,34,lower_inflation
3,2024-01-01,2024-01-01,123.9838,0.170393,2.887107,3,lower_inflation


In [10]:
# Summarize article volume and inflation levels by inflation regime.
summary_table = (
    analysis_df.groupby("inflation_regime")
    .agg(
        average_article_count=("article_count", "mean"),
        average_inflation_yoy=("inflation_yoy", "mean"),
        number_of_months=("month", "count"),
    )
    .round(2)
)

# Display the grouped summary table used for the second visualization.
summary_table

,average_article_count,average_inflation_yoy,number_of_months
inflation_regime,,,
high_inflation,30.00,3.79,1
lower_inflation,23.33,3.25,3


In [11]:
# Compute the correlation between inflation and media coverage.
correlation_value = (
    analysis_df[["inflation_yoy", "article_count"]]
    .corr()
    .iloc[0, 1]
)

# Print the correlation coefficient for interpretation.
print(
    "Correlation between inflation_yoy and article_count:",
    round(correlation_value, 3)
)

Correlation between inflation_yoy and article_count: 0.736


In [12]:
# Set a consistent visual style for all seaborn plots.
sns.set_theme(style="whitegrid")

In [13]:
# Install Plotly for interactive visualizations (required for advanced plots).
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [14]:
# Use browser renderer to reliably display Plotly figures outside the notebook.
import plotly.io as pio
pio.renderers.default = "browser"

In [15]:
# Create an interactive scatter plot to analyze the relationship
# between inflation and media coverage at the monthly level.
import plotly.express as px

fig = px.scatter(
    analysis_df,
    x="inflation_yoy",
    y="article_count",
    color="inflation_regime",
    size="article_count",
    hover_data=["month"],
    title="Relationship between Inflation and Media Coverage"
)

# Improve axis labels for presentation and interpretation.
fig.update_layout(
    xaxis_title="Inflation (Year-over-Year)",
    yaxis_title="Number of Articles",
)

# Display the interactive figure.
fig.show()

In [16]:
# Create an interactive dual-axis time series plot
# to compare inflation and media coverage over time.
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

fig = go.Figure()

# Add yearly inflation as the first time series on the left y-axis.
fig.add_trace(go.Scatter(
    x=analysis_df["month"],
    y=analysis_df["inflation_yoy"],
    mode="lines+markers",
    name="Inflation YoY",
    hovertemplate="Month: %{x}<br>Inflation YoY: %{y:.2f}%<extra></extra>"
))

# Add article counts as the second time series on the right y-axis.
fig.add_trace(go.Scatter(
    x=analysis_df["month"],
    y=analysis_df["article_count"],
    mode="lines+markers",
    name="Article Count",
    yaxis="y2",
    hovertemplate="Month: %{x}<br>Articles: %{y}<extra></extra>"
))

# Improve layout and show both variables on separate y-axes.
fig.update_layout(
    title="Interactive Time Series: Inflation and Media Coverage",
    xaxis_title="Month",
    yaxis=dict(title="Inflation YoY (%)"),
    yaxis2=dict(
        title="Number of Articles",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified"
)

fig.show()

In [1]:
import os
import pandas as pd

# Definiere den vollständigen Pfad (ersetze diesen Pfad durch den tatsächlichen Pfad auf deinem System)
csv_file_path = '/Users/ceydadut/Desktop/DataScienceProject/website/data/rq7_data.csv'

# Überprüfe, ob der Ordner existiert, andernfalls erstelle ihn
if not os.path.exists(os.path.dirname(csv_file_path)):
    os.makedirs(os.path.dirname(csv_file_path))

# Beispiel DataFrame für die CSV-Datei
rq7_df = pd.DataFrame({
    "month": ["2023-10-01", "2023-11-01", "2023-12-01", "2024-01-01"],
    "inflation_yoy": [3.79, 3.25, 3.71, 2.88],
    "article_count": [30, 33, 34, 3],
    "inflation_regime": ["high_inflation", "lower_inflation", "lower_inflation", "lower_inflation"]
})

# Speichern der CSV-Datei
rq7_df.to_csv(csv_file_path, index=False)

print(f"CSV-Datei wurde erfolgreich gespeichert unter: {csv_file_path}")

CSV-Datei wurde erfolgreich gespeichert unter: /Users/ceydadut/Desktop/DataScienceProject/website/data/rq7_data.csv
